# NQP / ATLAS — Phase 2: cross-architecture O_h on GPU (Colab)

Measures the inter-head residual overlap **O_h** on 7B-scale models (Mistral-7B, Llama-3.1-8B) that do **not** fit in local RAM. Forward-only, fp16 on a Colab T4 (16 GB).

**Method (frozen, see `docs/cross_architecture_plan.md`):** deepest layer, N=1200 residuals/head, `group_mode=query`, d_local from the model's own TwoNN, bootstrap CI over head pairs, and the **GQA intra/inter-group split** — the cross-arch comparand is the **inter-group** O_h (independent value spaces, apples-to-apples with GPT-2's 0.284 [0.276, 0.292]).

**Gate before trusting any 7B number — the fp16 regression:** we first run GPT-2 in fp16 on the GPU and confirm it reproduces O_h≈0.284. Subspace geometry should be precision-robust; this verifies it rather than assuming it. If fp16/GPU moved O_h, we'd know *before* believing Mistral.

**Runtime:** set `Runtime → Change runtime type → T4 GPU` before running.

## 1. Environment + repo

In [ ]:
import torch, sys
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
!pip -q install -U "transformers>=4.45" datasets accelerate

In [ ]:
# Clone the repo (public). If private, use a PAT or upload src/ manually.
REPO = 'https://github.com/jpcpol/ATLAS---Attention-s-Latent-Atlas-of-Specialized-heads-in-Transformers.git'
!git clone -q $REPO atlas || (cd atlas && git pull -q)
sys.path.insert(0, '/content/atlas/src')
print('src on path:', '/content/atlas/src')

## 2. HuggingFace token (from Colab secrets — never hardcode)

Add your token in the left sidebar: **🔑 Secrets → New secret**, name `HF_TOKEN`, value `hf_...`, and toggle *Notebook access*. Mistral-7B is ungated; Llama needs your approved access.

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami
login(userdata.get('HF_TOKEN'))
print('logged in as:', whoami()['name'])

## 3. Gate — GPT-2 fp16 regression on GPU

Must land on O_h ≈ 0.284 (frozen CPU/fp32 value). Confirms fp16/GPU does not move the subspace geometry before we trust the 7B runs.

In [ ]:
import atlas_crossarch as ax

# GPT-2 is MHA (n_rep=1): inter-group == global by construction.
gate = ax.run(model_name='gpt2', device='cuda', n_blocks=12, n_points=1200)
oh = gate['inter_O_h']
print(f"\nfp16/GPU GPT-2 inter O_h = {oh:.3f}  (frozen CPU/fp32 = 0.284)")
assert abs(oh - 0.284) < 0.01, 'fp16 moved O_h on GPT-2 — investigate before trusting 7B runs.'
print('GATE OK — fp16/GPU is geometry-preserving.')

## 4. Mistral-7B-v0.1 (ungated — runs now)

In [ ]:
res_mistral = ax.run(model_name='mistralai/Mistral-7B-v0.1', device='cuda',
                     n_blocks=12, n_points=1200)

## 5. Llama-3.1-8B (needs approved access)

Run only after Meta approves your request at huggingface.co/meta-llama/Llama-3.1-8B. Same code path; only the model id changes.

In [ ]:
res_llama = None
try:
    res_llama = ax.run(model_name='meta-llama/Llama-3.1-8B', device='cuda',
                       n_blocks=12, n_points=1200)
except Exception as e:
    print('Llama skipped (likely access pending):', type(e).__name__, str(e)[:160])

## 6. Collect + save results (download the JSON)

In [ ]:
import json, datetime

def summary(r):
    if r is None:
        return None
    lo, hi = r['inter_ci']
    return {'model': r['model'], 'family': r['geometry']['family'],
            'n_q': r['geometry']['n_q'], 'n_kv': r['geometry']['n_kv'],
            'd_local': r['d_local'], 'mean_TwoNN': round(r['mean_dim'], 2),
            'inter_O_h': round(r['inter_O_h'], 3), 'inter_CI': [round(lo,3), round(hi,3)],
            'dlocal_spread': round(r['dlocal_spread'], 3)}

out = {'date': datetime.date.today().isoformat(),
       'gpt2_fp16_gate': summary(gate),
       'mistral_7b': summary(res_mistral),
       'llama_8b': summary(res_llama),
       'gpt2_cpu_reference': {'inter_O_h': 0.284, 'inter_CI': [0.276, 0.292]},
       'qwen_0_5b_phase1': {'inter_O_h': 0.290, 'inter_CI': [0.284, 0.297]}}

print(json.dumps(out, indent=2))
with open('phase2_results.json', 'w') as f:
    json.dump(out, f, indent=2)
try:
    from google.colab import files; files.download('phase2_results.json')
except Exception:
    pass

## 7. Read the result (Gate G2)

- All inter-group O_h ≈ 0.27–0.29, CIs overlapping GPT-2 → **Case A** (architecture-independent).
- All ≪ 1 but spread → **Case B** (existence robust, magnitude architecture-dependent).
- Any ≈ 1 → **Case C** — treat as a GQA/RoPE extraction bug until verified by hand.

Paste `phase2_results.json` back into the project; it feeds `docs/cross_architecture_plan.md` Phase 2 and (only if Case A/B) the title/abstract promotion.